# NCKH — V7: Dual-Evidence Router

## Mục tiêu

V6 đã bảo vệ positive rất tốt nhưng lại tin `Qwen target_evidence=STRONG` quá mạnh, khiến một số hard-negative bị quay về CountGD++ baseline.

V7 **không chạy lại Qwen và CountGD++**. Nó dùng lại toàn bộ tín hiệu đã có trong V6:

- `target_evidence` từ Qwen router;
- `keep_boxes_shown / reject_boxes_shown` từ V6;
- `retention_ratio = V4_filtered_count / CountGD_baseline_count`;
- `decision=USE/NONE` từ V5 confounder discovery;
- `neg_text_count`;
- `neg_text_exemplar_count`;
- `v4_filtered_count`.

## Ý tưởng

```text
Qwen semantic evidence
        +
V4 detection survival evidence
        ↓
Dual-Evidence Router
```

### Router V7 đã freeze trước khi chạy

1. `baseline_count == 0` → giữ `BASELINE_ZERO`.
2. `target_evidence == NONE`
   - V5 tìm được confounder (`decision=USE`) → `NEG_TEXT_EXEMPLAR`
   - không tìm được confounder → `V4`
3. `target_evidence == STRONG`
   - nếu V6 cho thấy ít nhất `5/6` KEEP boxes **và** V4 vẫn giữ ít nhất `50%` CountGD detections → `BASELINE`
   - nếu không đạt consensus nhưng V5 có confounder → `NEG_TEXT`
   - còn lại → `V4`
4. Nếu sau này router có `WEAK` → consensus thì BASELINE, có confounder thì NEG_TEXT, còn lại V4.

## Scientific note

V7 vẫn là **exploratory follow-up** vì được thiết kế sau khi đã quan sát V4–V6 trên cùng benchmark 111 hard-negative + 96 positive. Không được gọi V7 là confirmatory final result cho đến khi chạy trên một held-out negative set mới.


In [ ]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================
from pathlib import Path
import random

SEED = 42
random.seed(SEED)
RUN_MODE = "FULL"   # "SMOKE" | "FULL"

MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS = 5
RETENTION_THRESHOLD = 0.50

SMOKE_HARD_N = 12
SMOKE_POS_N = 12

OUTPUT_DIR = Path("/kaggle/working/nckh_v7_dual_evidence_router")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

V7_RESULTS_CSV = OUTPUT_DIR / "v7_results.csv"
V7_METRICS_CSV = OUTPUT_DIR / "v7_metrics.csv"
V7_ORIGIN_CSV = OUTPUT_DIR / "v7_metrics_by_origin.csv"
V7_ROUTE_AUDIT_CSV = OUTPUT_DIR / "v7_route_audit.csv"
V7_FAILURES_CSV = OUTPUT_DIR / "v7_failure_cases.csv"
V7_BOOTSTRAP_JSON = OUTPUT_DIR / "v7_bootstrap.json"
V7_SUMMARY_JSON = OUTPUT_DIR / "v7_summary.json"

print("RUN_MODE:", RUN_MODE)
print("MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS:", MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS)
print("RETENTION_THRESHOLD:", RETENTION_THRESHOLD)


## Input cần Add vào Kaggle

V7 chỉ cần **kết quả V6**.

Có thể Add Input theo một trong hai dạng:

- `nckh_v6_guarded_router_export.zip`
- hoặc một Kaggle Dataset đã giải nén có:
  - `v6_results.csv`
  - `v6_router_predictions.csv`

**Không cần GPU, không cần Internet, không cần FSC-147, không cần chạy lại Qwen hay CountGD++.**


In [ ]:
# ============================================================
# CELL 2 — FIND / EXTRACT V6 INPUT
# ============================================================
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import json

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_file(name):
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob(name) if p.is_file()])
    if not hits:
        return None
    return sorted(hits, key=lambda p: (0 if str(p).startswith("/kaggle/input") else 1, len(str(p))))[0]

V6_ZIP = find_file("nckh_v6_guarded_router_export.zip")
V6_RESULTS_DIRECT = find_file("v6_results.csv")

if V6_ZIP is not None:
    V6_ROOT = Path("/kaggle/working/v6_export_for_v7")
    V6_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(V6_ZIP, "r") as zf:
        zf.extractall(V6_ROOT)
    print("Using V6 ZIP:", V6_ZIP)
elif V6_RESULTS_DIRECT is not None:
    V6_ROOT = V6_RESULTS_DIRECT.parent
    print("Using extracted V6 dataset:", V6_ROOT)
else:
    raise FileNotFoundError("Không tìm thấy nckh_v6_guarded_router_export.zip hoặc v6_results.csv")

V6_RESULTS = next(iter(V6_ROOT.rglob("v6_results.csv")), None)
V6_ROUTER = next(iter(V6_ROOT.rglob("v6_router_predictions.csv")), None)

for p in [V6_RESULTS, V6_ROUTER]:
    if p is None or not p.exists():
        raise FileNotFoundError(f"Thiếu file V6 cần thiết: {p}")

print("V6_RESULTS:", V6_RESULTS)
print("V6_ROUTER:", V6_ROUTER)


In [ ]:
# ============================================================
# CELL 3 — LOAD + VALIDATE DATA
# ============================================================
import pandas as pd
import numpy as np

v6 = pd.read_csv(V6_RESULTS)
router = pd.read_csv(V6_ROUTER)

required_v6 = {
    "group_id", "pair_type", "image_name", "target", "origin_batch",
    "gt_count", "baseline_count", "v4_filtered_count", "decision",
    "neg_text_count", "neg_text_exemplar_count", "target_evidence",
    "v6_count", "v5_hybrid_count",
}
required_router = {"group_id", "keep_boxes_shown", "reject_boxes_shown"}

missing = sorted(required_v6 - set(v6.columns))
if missing:
    raise ValueError("v6_results.csv thiếu cột: " + str(missing))
missing = sorted(required_router - set(router.columns))
if missing:
    raise ValueError("v6_router_predictions.csv thiếu cột: " + str(missing))

df = v6.merge(
    router[["group_id", "keep_boxes_shown", "reject_boxes_shown"]],
    on="group_id", how="left"
)

numeric_cols = [
    "gt_count", "baseline_count", "v4_filtered_count", "neg_text_count",
    "neg_text_exemplar_count", "v6_count", "v5_hybrid_count",
    "keep_boxes_shown", "reject_boxes_shown",
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
if df[numeric_cols].isna().any().any():
    raise ValueError("Có giá trị numeric bị thiếu sau khi merge V6 results + router.")

df["retention_ratio"] = np.where(
    df["baseline_count"] > 0,
    df["v4_filtered_count"] / df["baseline_count"],
    1.0,
)

df["target_consensus"] = (
    (df["keep_boxes_shown"] >= MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS)
    & (df["retention_ratio"] >= RETENTION_THRESHOLD)
).astype(int)

print("Rows:", len(df))
print(df["pair_type"].value_counts())
print("\nTarget evidence:")
print(pd.crosstab(df["pair_type"], df["target_evidence"]))
print("\nV5 discovery decision:")
print(pd.crosstab(df["pair_type"], df["decision"]))
print("\nTarget consensus:")
print(pd.crosstab(df["pair_type"], df["target_consensus"]))

display(df[[
    "group_id", "pair_type", "target", "baseline_count", "v4_filtered_count",
    "retention_ratio", "target_evidence", "keep_boxes_shown",
    "reject_boxes_shown", "decision", "target_consensus"
]].head(12))


In [ ]:
# ============================================================
# CELL 4 — OPTIONAL SMOKE SUBSET
# ============================================================
if RUN_MODE == "SMOKE":
    hard = df[df["pair_type"].eq("HARD_NEGATIVE")].copy()
    pos = df[df["pair_type"].eq("POSITIVE")].copy()
    hard_run = hard.sample(n=min(SMOKE_HARD_N, len(hard)), random_state=SEED)
    pos_run = pos.sample(n=min(SMOKE_POS_N, len(pos)), random_state=SEED + 1)
    run_df = pd.concat([hard_run, pos_run], ignore_index=True)
else:
    run_df = df.copy()

run_df = run_df.sort_values(["pair_type", "origin_batch", "group_id"]).reset_index(drop=True)
print("THIS RUN:")
print(run_df["pair_type"].value_counts())


In [ ]:
# ============================================================
# CELL 5 — FROZEN V7 ROUTER
# ============================================================
def route_v7(row):
    baseline = int(row["baseline_count"])
    v4 = int(row["v4_filtered_count"])
    neg_text = int(row["neg_text_count"])
    neg_ex = int(row["neg_text_exemplar_count"])

    target_evidence = str(row["target_evidence"]).upper().strip()
    v5_use = str(row["decision"]).upper().strip() == "USE"
    consensus = bool(row["target_consensus"])

    if baseline == 0:
        return "BASELINE_ZERO", 0, "baseline_already_zero"

    if target_evidence == "NONE":
        if v5_use:
            return "NEG_TEXT_EXEMPLAR", neg_ex, "target_none_and_v5_confounder_use"
        return "V4", v4, "target_none_no_reliable_confounder"

    if target_evidence == "STRONG":
        if consensus:
            return "BASELINE", baseline, "strong_target_plus_detection_consensus"
        if v5_use:
            return "NEG_TEXT", neg_text, "strong_target_but_low_detection_consensus_with_confounder"
        return "V4", v4, "strong_target_but_low_detection_consensus_no_confounder"

    if consensus:
        return "BASELINE", baseline, "weak_or_unknown_but_detection_consensus"
    if v5_use:
        return "NEG_TEXT", neg_text, "weak_or_unknown_with_confounder"
    return "V4", v4, "weak_or_unknown_fallback_v4"

routes = run_df.apply(route_v7, axis=1)
run_df["v7_route"] = [x[0] for x in routes]
run_df["v7_count"] = [int(x[1]) for x in routes]
run_df["v7_reason"] = [x[2] for x in routes]
run_df.to_csv(V7_RESULTS_CSV, index=False)

print("=== V7 ROUTES ===")
print(pd.crosstab(run_df["pair_type"], run_df["v7_route"]))

positive_exemplar = int((
    run_df["pair_type"].eq("POSITIVE")
    & run_df["v7_route"].eq("NEG_TEXT_EXEMPLAR")
).sum())
print("\nPositive exemplar route count:", positive_exemplar)

display(run_df[[
    "group_id", "pair_type", "target", "target_evidence", "retention_ratio",
    "keep_boxes_shown", "decision", "v7_route", "v7_count", "v7_reason"
]].head(20))


In [ ]:
# ============================================================
# CELL 6 — MAIN METRICS
# ============================================================
import pandas as pd
import numpy as np

methods = {
    "COUNTGD_BASELINE": "baseline_count",
    "V4_BOX_FILTER": "v4_filtered_count",
    "V5_NEG_TEXT": "neg_text_count",
    "V5_NEG_TEXT_PLUS_EXEMPLAR": "neg_text_exemplar_count",
    "V5_HYBRID_EXPLORATORY": "v5_hybrid_count",
    "V6_GUARDED_ROUTER": "v6_count",
    "V7_DUAL_EVIDENCE_ROUTER": "v7_count",
}

rows = []
for pair_type, g in run_df.groupby("pair_type"):
    for method, col in methods.items():
        y = g["gt_count"].to_numpy(dtype=float)
        p = g[col].to_numpy(dtype=float)
        row = {
            "pair_type": pair_type,
            "method": method,
            "n": int(len(g)),
            "mae": float(np.mean(np.abs(p - y))),
            "rmse": float(np.sqrt(np.mean((p - y) ** 2))),
            "mean_gt": float(np.mean(y)),
            "mean_pred": float(np.mean(p)),
        }
        if pair_type == "HARD_NEGATIVE":
            row["false_activation_rate"] = float(np.mean(p > 0))
            row["zero_rate"] = float(np.mean(p == 0))
            row["positive_survival_rate"] = np.nan
        else:
            row["false_activation_rate"] = np.nan
            row["zero_rate"] = np.nan
            row["positive_survival_rate"] = float(np.mean(p > 0))
        rows.append(row)

metrics = pd.DataFrame(rows)
metrics.to_csv(V7_METRICS_CSV, index=False)
print("=== MAIN METRICS ===")
display(metrics)
print("\n=== V7 ONLY ===")
display(metrics[metrics["method"].eq("V7_DUAL_EVIDENCE_ROUTER")])


In [ ]:
# ============================================================
# CELL 7 — OLD_56 vs NEW_55
# ============================================================
origin_rows = []
hard = run_df[run_df["pair_type"].eq("HARD_NEGATIVE")].copy()

for origin, g in hard.groupby("origin_batch"):
    for method, col in methods.items():
        p = g[col].to_numpy(dtype=float)
        origin_rows.append({
            "origin_batch": origin,
            "method": method,
            "n": int(len(g)),
            "mae": float(np.mean(np.abs(p))),
            "rmse": float(np.sqrt(np.mean(p ** 2))),
            "fpr": float(np.mean(p > 0)),
            "zero_rate": float(np.mean(p == 0)),
            "mean_pred": float(np.mean(p)),
        })

origin_metrics = pd.DataFrame(origin_rows)
origin_metrics.to_csv(V7_ORIGIN_CSV, index=False)
print("=== V7 BY HARD-NEGATIVE ORIGIN ===")
display(origin_metrics[origin_metrics["method"].eq("V7_DUAL_EVIDENCE_ROUTER")])


In [ ]:
# ============================================================
# CELL 8 — PAIRED BOOTSTRAP
# ============================================================
import numpy as np
import json
rng = np.random.default_rng(SEED)

def bootstrap_diff(a, b, n_boot=5000):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) != len(b):
        raise ValueError("Paired arrays must have equal length.")
    n = len(a)
    point = float(np.mean(a - b))
    vals = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals[i] = float(np.mean(a[idx] - b[idx]))
    lo, hi = np.quantile(vals, [0.025, 0.975])
    return {
        "n": int(n),
        "point_estimate_A_minus_B": point,
        "ci95_low": float(lo),
        "ci95_high": float(hi),
    }

hard = run_df[run_df["pair_type"].eq("HARD_NEGATIVE")].copy()
pos = run_df[run_df["pair_type"].eq("POSITIVE")].copy()

boot = {
    "interpretation": "Negative means V7 has lower absolute error than comparator.",
    "hard_V7_minus_V4": bootstrap_diff(
        np.abs(hard["v7_count"] - hard["gt_count"]),
        np.abs(hard["v4_filtered_count"] - hard["gt_count"]),
    ),
    "hard_V7_minus_V6": bootstrap_diff(
        np.abs(hard["v7_count"] - hard["gt_count"]),
        np.abs(hard["v6_count"] - hard["gt_count"]),
    ),
    "hard_V7_minus_baseline": bootstrap_diff(
        np.abs(hard["v7_count"] - hard["gt_count"]),
        np.abs(hard["baseline_count"] - hard["gt_count"]),
    ),
    "positive_V7_minus_V4": bootstrap_diff(
        np.abs(pos["v7_count"] - pos["gt_count"]),
        np.abs(pos["v4_filtered_count"] - pos["gt_count"]),
    ),
    "positive_V7_minus_V6": bootstrap_diff(
        np.abs(pos["v7_count"] - pos["gt_count"]),
        np.abs(pos["v6_count"] - pos["gt_count"]),
    ),
    "positive_V7_minus_baseline": bootstrap_diff(
        np.abs(pos["v7_count"] - pos["gt_count"]),
        np.abs(pos["baseline_count"] - pos["gt_count"]),
    ),
}
with open(V7_BOOTSTRAP_JSON, "w", encoding="utf-8") as f:
    json.dump(boot, f, indent=2)
print(json.dumps(boot, indent=2))


In [ ]:
# ============================================================
# CELL 9 — ROUTE AUDIT + FAILURE CASES
# ============================================================
audit_cols = [
    "group_id", "pair_type", "image_name", "source_class", "target",
    "origin_batch", "gt_count", "baseline_count", "v4_filtered_count",
    "retention_ratio", "target_evidence", "keep_boxes_shown",
    "reject_boxes_shown", "decision", "confounder", "neg_text_count",
    "neg_text_exemplar_count", "v6_count", "v7_route", "v7_count", "v7_reason",
]
audit_cols = [c for c in audit_cols if c in run_df.columns]
run_df[audit_cols].to_csv(V7_ROUTE_AUDIT_CSV, index=False)

run_df["baseline_abs_error"] = (run_df["baseline_count"] - run_df["gt_count"]).abs()
run_df["v4_abs_error"] = (run_df["v4_filtered_count"] - run_df["gt_count"]).abs()
run_df["v6_abs_error"] = (run_df["v6_count"] - run_df["gt_count"]).abs()
run_df["v7_abs_error"] = (run_df["v7_count"] - run_df["gt_count"]).abs()
run_df["v7_vs_v4_improvement"] = run_df["v4_abs_error"] - run_df["v7_abs_error"]
run_df["v7_vs_v6_improvement"] = run_df["v6_abs_error"] - run_df["v7_abs_error"]

failure_cols = [
    "group_id", "pair_type", "image_name", "source_class", "target",
    "origin_batch", "gt_count", "target_evidence", "retention_ratio",
    "keep_boxes_shown", "decision", "confounder", "baseline_count",
    "v4_filtered_count", "neg_text_count", "neg_text_exemplar_count",
    "v6_count", "v7_route", "v7_count", "baseline_abs_error",
    "v4_abs_error", "v6_abs_error", "v7_abs_error",
    "v7_vs_v4_improvement", "v7_vs_v6_improvement",
]
failure_cols = [c for c in failure_cols if c in run_df.columns]

failures = pd.concat([
    run_df.sort_values("v7_vs_v4_improvement").head(30),
    run_df.sort_values("v7_vs_v4_improvement", ascending=False).head(30),
], ignore_index=True).drop_duplicates("group_id")
failures[failure_cols].to_csv(V7_FAILURES_CSV, index=False)

print("=== MOST DEGRADED VS V4 ===")
display(run_df.sort_values("v7_vs_v4_improvement")[failure_cols].head(15))
print("\n=== MOST IMPROVED VS V4 ===")
display(run_df.sort_values("v7_vs_v4_improvement", ascending=False)[failure_cols].head(15))


In [ ]:
# ============================================================
# CELL 10 — SUMMARY + EXPORT ZIP
# ============================================================
import json
import shutil
from pathlib import Path
from IPython.display import FileLink, display

v7_metric_rows = metrics[metrics["method"].eq("V7_DUAL_EVIDENCE_ROUTER")].to_dict("records")

summary = {
    "phase": "V7_DUAL_EVIDENCE_ROUTER",
    "run_mode": RUN_MODE,
    "n_groups": int(len(run_df)),
    "pair_type_counts": run_df["pair_type"].value_counts().to_dict(),
    "frozen_router": {
        "min_keep_boxes_for_target_consensus": int(MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS),
        "retention_threshold": float(RETENTION_THRESHOLD),
        "rule": {
            "NONE+USE": "NEG_TEXT_EXEMPLAR",
            "NONE+NONE": "V4",
            "STRONG+consensus": "BASELINE",
            "STRONG+no_consensus+USE": "NEG_TEXT",
            "STRONG+no_consensus+NONE": "V4",
            "WEAK_or_unknown+consensus": "BASELINE",
            "WEAK_or_unknown+USE": "NEG_TEXT",
            "WEAK_or_unknown+NONE": "V4",
        },
    },
    "route_counts": {k: int(v) for k, v in run_df["v7_route"].value_counts().to_dict().items()},
    "route_counts_by_pair_type": {
        pair_type: {k: int(v) for k, v in g["v7_route"].value_counts().to_dict().items()}
        for pair_type, g in run_df.groupby("pair_type")
    },
    "positive_exemplar_route_count": int((
        run_df["pair_type"].eq("POSITIVE")
        & run_df["v7_route"].eq("NEG_TEXT_EXEMPLAR")
    ).sum()),
    "v7_metrics": v7_metric_rows,
    "bootstrap": boot,
    "scientific_notes": [
        "V7 is exploratory because it was designed after observing V4-V6 on the same benchmark.",
        "The primary router uses fixed 5/6 KEEP-box consensus and 0.50 V4 retention threshold; no threshold grid search is performed in this notebook.",
        "V7 reuses cached V4/V5/V6 outputs and does not rerun Qwen or CountGD++.",
        "A fresh held-out negative benchmark is required before confirmatory claims.",
    ],
}

with open(V7_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

archive = shutil.make_archive(
    "/kaggle/working/nckh_v7_dual_evidence_router_export",
    "zip",
    root_dir=str(OUTPUT_DIR),
)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nCreated:", archive)
print("Size MB:", round(Path(archive).stat().st_size / 1024**2, 3))
display(FileLink(archive))


## Cách đọc V7

Ưu tiên các file:

- `v7_summary.json`
- `v7_metrics.csv`
- `v7_metrics_by_origin.csv`
- `v7_bootstrap.json`
- `v7_route_audit.csv`
- `v7_failure_cases.csv`

### Điều kiện V7 có tín hiệu tốt

So với V4:
- `Hard MAE < 14.96`
- `Hard FPR < 23.42%`
- `Positive MAE <= 16.71`

Đặc biệt xem `positive_exemplar_route_count`; lý tưởng là `0`.
